# Notebook 03 — Testy Statystyczne

**Projekt:** Współzależność rynków aktywów cyfrowych i tradycyjnych systemów finansowych  

---

## Cel notebooka

Przeprowadzamy formalne testy statystyczne wymagane do uzasadnienia metodologicznego modeli ML:

1. **Stacjonarność** — ADF i KPSS (warunek dla modeli szeregów czasowych)
2. **Autokorelacja** — ACF/PACF zwrotów i kwadratów zwrotów (efekty ARCH)
3. **Normalność** — Jarque-Bera, Shapiro-Wilk (potwierdzenie gruboogonowości)
4. **Grube ogony** — estymator Hilla (indeks alfa Pareto)
5. **Heteroskedastyczność** — test LM Engla (efekty ARCH)
6. **Model GARCH(1,1)** — warunkowa zmienność BTC i WIG20
7. **Kointegracja Johansena** — długookresowe powiązanie BTC–WIG20
8. **Przyczynowość Grangera** — kierunek transmisji ryzyka

## Metodologiczne uzasadnienie
- Testy stacjonarności: modele LSTM wymagają stacjonarnych danych wejściowych (log-returns spełniają ten warunek)
- Grube ogony potwierdzają konieczność użycia modeli odpornych na ekstrema
- Test Grangera formalnie testuje kierunek transmisji BTC → WIG20

In [1]:
# ============================================================
# Reprodukowalność + Importy
# ============================================================
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
import matplotlib.gridspec as gridspec
import seaborn as sns
import warnings
from pathlib import Path
from scipy import stats
from statsmodels.tsa.stattools import adfuller, kpss, acf, pacf, grangercausalitytests, coint
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.stats.diagnostic import het_arch
from statsmodels.tsa.vector_ar.vecm import coint_johansen
from arch import arch_model

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
warnings.filterwarnings('ignore')

# ── Ścieżki ─────────────────────────────────────────────────
BASE_DIR = Path('.')
DATA_PROC = BASE_DIR / 'data' / 'processed'
RESULTS_TABLES = BASE_DIR / 'results' / 'tables'
RESULTS_FIGS = BASE_DIR / 'results' / 'figures'

for d in [RESULTS_TABLES, RESULTS_FIGS]:
    d.mkdir(parents=True, exist_ok=True)

# ── Styl wykresów ────────────────────────────────────────────
plt.style.use('seaborn-v0_8-whitegrid')
mpl.rcParams['font.family'] = 'DejaVu Sans'
mpl.rcParams['font.size'] = 11
mpl.rcParams['axes.titlesize'] = 13
mpl.rcParams['figure.dpi'] = 150
mpl.rcParams['savefig.dpi'] = 300

COLOR_BTC = '#F7931A'
COLOR_WIG = '#003087'
COLOR_HRI = '#C0392B'

print("Biblioteki zaladowane.")
print(f"statsmodels: v{__import__('statsmodels').__version__}")
print(f"arch: v{__import__('arch').__version__}")

Biblioteki zaladowane.
statsmodels: v0.14.4
arch: v8.0.0


In [2]:
# ── Wczytanie danych ─────────────────────────────────────────
master_path = DATA_PROC / 'master_dataset.csv'
df = pd.read_csv(master_path, index_col='Date', parse_dates=True)

# Oblicz log-returns jeśli nie ma
if 'btc_log_return' not in df.columns:
    df['btc_log_return'] = np.log(df['btc_close'] / df['btc_close'].shift(1))
if 'wig_log_return' not in df.columns:
    df['wig_log_return'] = np.log(df['wig_close'] / df['wig_close'].shift(1))

print(f"Dane wczytane: {df.shape}, zakres: {df.index.min().date()} → {df.index.max().date()}")
print(f"Kolumny: {list(df.columns)}")

Dane wczytane: (1906, 24), zakres: 2019-01-02 → 2026-08-14
Kolumny: ['wig_open', 'wig_high', 'wig_low', 'wig_close', 'wig_volume', 'btc_open', 'btc_high', 'btc_low', 'btc_close', 'btc_volume', 'vix', 'dxy', 'fed_rate', 'flag_btc_outlier', 'flag_wig_outlier', 'btc_log_return', 'wig_log_return', 'vix_change', 'dxy_change', 'btc_1h_return_mean', 'btc_1h_return_std', 'btc_1h_return_min', 'btc_1h_return_max', 'btc_1h_n_obs']


## 1. Testy stacjonarności (ADF i KPSS)

**ADF (Augmented Dickey-Fuller):**
- H₀: szereg ma pierwiastek jednostkowy (niestacjonarny)
- Odrzucamy H₀ (p < 0.05) → szereg jest stacjonarny

**KPSS (Kwiatkowski–Phillips–Schmidt–Shin):**
- H₀: szereg jest stacjonarny
- Nie odrzucamy H₀ (p > 0.05) → stacjonarny

Oczekiwania: poziomy cen — I(1), log-zwroty — I(0) (stacjonarne)

In [3]:
# ============================================================
# KROK 1: Testy stacjonarności ADF i KPSS
# ============================================================

def run_adf_test(series, series_name, regression='c'):
    """
    Przeprowadza test ADF i zwraca wyniki jako słownik.
    
    Parameters
    ----------
    series : pd.Series
        Szereg czasowy
    series_name : str
        Nazwa szeregu
    regression : str
        'c' = ze stałą, 'ct' = ze stałą i trendem, 'nc' = bez stałej
    """
    clean = series.dropna()
    result = adfuller(clean, autolag='AIC', regression=regression)
    
    return {
        'Szereg': series_name,
        'ADF stat': round(result[0], 4),
        'ADF p-value': round(result[1], 6),
        'ADF lagi': result[2],
        'ADF 1% CV': round(result[4]['1%'], 4),
        'ADF 5% CV': round(result[4]['5%'], 4),
        'ADF wniosek': 'Stacjonarny' if result[1] < 0.05 else 'Niestacjonarny (pierwiastek jedn.)'
    }

def run_kpss_test(series, series_name, regression='c'):
    """
    Przeprowadza test KPSS i zwraca wyniki jako słownik.
    """
    clean = series.dropna()
    # KPSS może rzucać UserWarning przy wielu lagach — ignorujemy
    with warnings.catch_warnings():
        warnings.simplefilter('ignore')
        result = kpss(clean, regression=regression, nlags='auto')
    
    return {
        'Szereg': series_name,
        'KPSS stat': round(result[0], 4),
        'KPSS p-value': round(result[1], 6),
        'KPSS lagi': result[2],
        'KPSS 5% CV': round(result[3]['5%'], 4),
        'KPSS wniosek': 'Niestacjonarny' if result[1] < 0.05 else 'Stacjonarny'
    }

# Szeregi do testowania
series_to_test = [
    (df['btc_close'],          'BTC/USD Close (poziom)'),
    (df['btc_log_return'],     'BTC/USD Log-Return'),
    (df['wig_close'],          'WIG20 Close (poziom)'),
    (df['wig_log_return'],     'WIG20 Log-Return'),
]
if 'vix' in df.columns:
    series_to_test.append((df['vix'], 'VIX (poziom)'))
if 'dxy' in df.columns:
    series_to_test.append((df['dxy'], 'DXY (poziom)'))

print("Przeprowadzanie testów ADF i KPSS...")
adf_results = []
kpss_results = []

for series, name in series_to_test:
    print(f"  Testowanie: {name}")
    
    # Wybierz regression: 'ct' dla poziomów (zakłada trend), 'c' dla zwrotów
    reg = 'ct' if 'poziom' in name.lower() else 'c'
    kpss_reg = 'ct' if 'poziom' in name.lower() else 'c'
    
    adf_res = run_adf_test(series, name, regression=reg)
    kpss_res = run_kpss_test(series, name, regression=kpss_reg)
    
    adf_results.append(adf_res)
    kpss_results.append(kpss_res)

# Zbuduj tabelę zbiorczą
adf_df = pd.DataFrame(adf_results).set_index('Szereg')
kpss_df = pd.DataFrame(kpss_results).set_index('Szereg')

# Połącz ADF i KPSS
stationarity_table = pd.DataFrame()
stationarity_table['ADF stat'] = adf_df['ADF stat']
stationarity_table['ADF p-val'] = adf_df['ADF p-value']
stationarity_table['ADF wniosek'] = adf_df['ADF wniosek']
stationarity_table['KPSS stat'] = kpss_df['KPSS stat']
stationarity_table['KPSS p-val'] = kpss_df['KPSS p-value']
stationarity_table['KPSS wniosek'] = kpss_df['KPSS wniosek']

# Łączny wniosek
def combined_conclusion(adf_w, kpss_w):
    is_adf_stat = 'Stacjonarny' in adf_w
    is_kpss_stat = 'Stacjonarny' in kpss_w
    if is_adf_stat and is_kpss_stat:
        return 'I(0) — stacjonarny'
    elif not is_adf_stat and not is_kpss_stat:
        return 'I(1) — niestacjonarny'
    else:
        return 'Nieokreślony (sprzeczne testy)'

stationarity_table['Wniosek łączny'] = [
    combined_conclusion(row['ADF wniosek'], row['KPSS wniosek'])
    for _, row in stationarity_table.iterrows()
]

print("\n" + "=" * 90)
print("WYNIKI TESTÓW STACJONARNOŚCI")
print("=" * 90)
print(stationarity_table[['ADF stat', 'ADF p-val', 'KPSS stat', 'KPSS p-val', 'Wniosek łączny']].to_string())

# Zapis
csv_path = RESULTS_TABLES / '03_stationarity_tests.csv'
stationarity_table.to_csv(csv_path, float_format='%.6f')
print(f"\nZapisano: {csv_path}")

tex_path = RESULTS_TABLES / '03_stationarity_tests.tex'
stationarity_table[['ADF stat', 'ADF p-val', 'KPSS stat', 'KPSS p-val', 'Wniosek łączny']].to_latex(
    str(tex_path), float_format='%.4f', escape=False,
    caption='Wyniki testów stacjonarności ADF i KPSS (2019--2024)',
    label='tab:stationarity_tests'
)
print(f"Zapisano LaTeX: {tex_path}")

Przeprowadzanie testów ADF i KPSS...
  Testowanie: BTC/USD Close (poziom)
  Testowanie: BTC/USD Log-Return
  Testowanie: WIG20 Close (poziom)
  Testowanie: WIG20 Log-Return
  Testowanie: VIX (poziom)
  Testowanie: DXY (poziom)

WYNIKI TESTÓW STACJONARNOŚCI
                        ADF stat  ADF p-val  KPSS stat  KPSS p-val                  Wniosek łączny
Szereg                                                                                            
BTC/USD Close (poziom)   -1.8102   0.699825     0.4529        0.01           I(1) — niestacjonarny
BTC/USD Log-Return      -24.0832   0.000000     0.2375        0.10              I(0) — stacjonarny
WIG20 Close (poziom)     -0.7887   0.966620     1.1452        0.01           I(1) — niestacjonarny
WIG20 Log-Return        -24.3402   0.000000     0.3003        0.10              I(0) — stacjonarny
VIX (poziom)             -5.1266   0.000120     0.2193        0.01  Nieokreślony (sprzeczne testy)
DXY (poziom)             -2.0937   0.549614     0.

## 2. Autokorelacja (ACF i PACF)

ACF i PACF dla zwrotów BTC i WIG20 oraz ich kwadratów. Kwadraty zwrotów wykazują silną autokorelację → efekty ARCH (skupianie zmienności).

In [4]:
# ============================================================
# KROK 2: ACF i PACF — zwroty i kwadraty
# ============================================================

LAGS = 40

btc_r = df['btc_log_return'].dropna()
wig_r = df['wig_log_return'].dropna()

fig, axes = plt.subplots(4, 2, figsize=(16, 18))
fig.suptitle('ACF i PACF — Log-zwroty i kwadraty log-zwrotów (efekty ARCH)',
             fontsize=13, y=1.01)

# Panel 1: ACF BTC returns
plot_acf(btc_r, ax=axes[0, 0], lags=LAGS, alpha=0.05,
         color=COLOR_BTC, title='BTC log-return — ACF')

# Panel 2: PACF BTC returns
plot_pacf(btc_r, ax=axes[0, 1], lags=LAGS, alpha=0.05,
          color=COLOR_BTC, title='BTC log-return — PACF')

# Panel 3: ACF WIG20 returns
plot_acf(wig_r, ax=axes[1, 0], lags=LAGS, alpha=0.05,
         color=COLOR_WIG, title='WIG20 log-return — ACF')

# Panel 4: PACF WIG20 returns
plot_pacf(wig_r, ax=axes[1, 1], lags=LAGS, alpha=0.05,
          color=COLOR_WIG, title='WIG20 log-return — PACF')

# Panel 5-8: Kwadraty zwrotów (efekty ARCH)
btc_r2 = btc_r ** 2
wig_r2 = wig_r ** 2

plot_acf(btc_r2, ax=axes[2, 0], lags=LAGS, alpha=0.05,
         color=COLOR_BTC, title='BTC (log-return)² — ACF [efekt ARCH]')
plot_pacf(btc_r2, ax=axes[2, 1], lags=LAGS, alpha=0.05,
          color=COLOR_BTC, title='BTC (log-return)² — PACF [efekt ARCH]')
plot_acf(wig_r2, ax=axes[3, 0], lags=LAGS, alpha=0.05,
         color=COLOR_WIG, title='WIG20 (log-return)² — ACF [efekt ARCH]')
plot_pacf(wig_r2, ax=axes[3, 1], lags=LAGS, alpha=0.05,
          color=COLOR_WIG, title='WIG20 (log-return)² — PACF [efekt ARCH]')

# Adnotacje
for ax in axes[2:, :].flat:
    ax.text(0.97, 0.95, 'Silna autokorelacja = efekt ARCH',
            transform=ax.transAxes, fontsize=8, ha='right', va='top',
            color='red', alpha=0.7)

plt.tight_layout()
for ext in ['png', 'pdf']:
    plt.savefig(RESULTS_FIGS / f'fig_08_acf_pacf.{ext}',
                dpi=300 if ext=='png' else 150, bbox_inches='tight')
plt.show()
print("Zapisano: fig_08_acf_pacf.png + .pdf")

# Ljung-Box test
from statsmodels.stats.diagnostic import acorr_ljungbox

print("\nTest Ljung-Box na autokorelację zwrotów:")
lb_lags = [1, 5, 10, 20]
lb_btc = acorr_ljungbox(btc_r, lags=lb_lags, return_df=True)
lb_wig = acorr_ljungbox(wig_r, lags=lb_lags, return_df=True)
lb_btc2 = acorr_ljungbox(btc_r2, lags=lb_lags, return_df=True)
lb_wig2 = acorr_ljungbox(wig_r2, lags=lb_lags, return_df=True)

lb_summary = pd.DataFrame({
    'BTC ret LB stat': lb_btc['lb_stat'].values,
    'BTC ret LB p-val': lb_btc['lb_pvalue'].values,
    'WIG ret LB stat': lb_wig['lb_stat'].values,
    'WIG ret LB p-val': lb_wig['lb_pvalue'].values,
    'BTC ret² LB p-val': lb_btc2['lb_pvalue'].values,
    'WIG ret² LB p-val': lb_wig2['lb_pvalue'].values,
}, index=lb_lags)
lb_summary.index.name = 'Lag'

print(lb_summary.round(4).to_string())
lb_summary.to_csv(RESULTS_TABLES / '03_ljungbox_test.csv', float_format='%.6f')
print("\nZapisano: 03_ljungbox_test.csv")

Zapisano: fig_08_acf_pacf.png + .pdf

Test Ljung-Box na autokorelację zwrotów:
     BTC ret LB stat  BTC ret LB p-val  WIG ret LB stat  WIG ret LB p-val  BTC ret² LB p-val  WIG ret² LB p-val
Lag                                                                                                            
1             5.4990            0.0190           1.9044            0.1676             0.0017                0.0
5            11.2820            0.0461           9.8536            0.0795             0.0000                0.0
10           14.2731            0.1609          14.7968            0.1396             0.0000                0.0
20           32.8649            0.0349          29.6440            0.0758             0.0026                0.0

Zapisano: 03_ljungbox_test.csv


## 3. Testy normalności

In [5]:
# ============================================================
# KROK 3: Testy normalności — JB, Shapiro-Wilk
# ============================================================

print("=" * 70)
print("TESTY NORMALNOŚCI")
print("=" * 70)

normality_results = []

for series, name in [
    (df['btc_log_return'].dropna(), 'BTC Log-Return'),
    (df['wig_log_return'].dropna(), 'WIG20 Log-Return'),
]:
    # Jarque-Bera
    jb_stat, jb_p = stats.jarque_bera(series)
    
    # Shapiro-Wilk (max 5000 obs dla wydajności)
    sample_size = min(5000, len(series))
    sample = series.sample(n=sample_size, random_state=SEED)
    sw_stat, sw_p = stats.shapiro(sample)
    
    # Kurtoza i skośność
    kurt = stats.kurtosis(series)  # excess kurtosis
    skewness = stats.skew(series)
    
    # Anderson-Darling
    ad_result = stats.anderson(series, dist='norm')
    ad_stat = ad_result.statistic
    ad_cv_5pct = ad_result.critical_values[2]  # 5% poziom
    ad_rejected = 'Tak (nie-normalny)' if ad_stat > ad_cv_5pct else 'Nie (normalny)'
    
    normality_results.append({
        'Szereg': name,
        'N': len(series),
        'JB stat': round(jb_stat, 2),
        'JB p-value': round(jb_p, 8),
        'SW stat (próbka 5k)': round(sw_stat, 6),
        'SW p-value': round(sw_p, 8),
        'AD stat': round(ad_stat, 4),
        'AD rej. H0 (5%)': ad_rejected,
        'Kurtoza nadmiarowa': round(kurt, 4),
        'Skośność': round(skewness, 4),
        'Wniosek': 'NIE-normalny' if jb_p < 0.05 else 'Normalny'
    })
    
    print(f"\n{name} (n={len(series)}):")
    print(f"  Kurtoza nadmiarowa: {kurt:.4f}")
    print(f"  Skośność:           {skewness:.4f}")
    print(f"  JB stat:            {jb_stat:.2f}, p = {jb_p:.2e}")
    print(f"  SW stat:            {sw_stat:.6f}, p = {sw_p:.2e} (próbka {sample_size})")
    print(f"  AD stat:            {ad_stat:.4f} (CV 5% = {ad_cv_5pct:.4f})")
    print(f"  Wniosek:            Rozkład JEST {'normalny' if jb_p > 0.05 else 'NIE-normalny (leptokurtyczny)'}")

norm_df = pd.DataFrame(normality_results).set_index('Szereg')

csv_path = RESULTS_TABLES / '03_normality_tests.csv'
norm_df.to_csv(csv_path, float_format='%.8f')
print(f"\nZapisano: {csv_path}")

tex_path = RESULTS_TABLES / '03_normality_tests.tex'
cols_to_show = ['JB stat', 'JB p-value', 'SW stat (próbka 5k)', 'SW p-value',
                'Kurtoza nadmiarowa', 'Skośność', 'Wniosek']
norm_df[cols_to_show].to_latex(
    str(tex_path), float_format='%.4f', escape=False,
    caption='Wyniki testów normalności rozkładu log-zwrotów',
    label='tab:normality_tests'
)
print(f"Zapisano LaTeX: {tex_path}")

TESTY NORMALNOŚCI

BTC Log-Return (n=1905):
  Kurtoza nadmiarowa: 16.6820
  Skośność:           -1.0505
  JB stat:            22439.55, p = 0.00e+00
  SW stat:            0.894964, p = 3.28e-34 (próbka 1905)
  AD stat:            36.3156 (CV 5% = 0.7850)
  Wniosek:            Rozkład JEST NIE-normalny (leptokurtyczny)

WIG20 Log-Return (n=1905):
  Kurtoza nadmiarowa: 8.0924
  Skośność:           -0.7084
  JB stat:            5357.39, p = 0.00e+00
  SW stat:            0.946111, p = 7.77e-26 (próbka 1905)
  AD stat:            10.7896 (CV 5% = 0.7850)
  Wniosek:            Rozkład JEST NIE-normalny (leptokurtyczny)

Zapisano: results/tables/03_normality_tests.csv
Zapisano LaTeX: results/tables/03_normality_tests.tex


## 4. Grube ogony — Estymator Hilla

Estymator Hilla szacuje indeks ogona α rozkładu Pareto. Im mniejsze α, tym grubsze ogony.
- α < 2: nieskończona wariancja (ekstremalnie grube ogony)
- α ∈ (2,4): skończona wariancja, nieskończona kurtoza
- α > 4: istnieje kurtoza

In [6]:
# ============================================================
# KROK 4: Estymator Hilla — grube ogony
# ============================================================

def hill_estimator(series, k_values=None):
    """
    Oblicza estymator Hilla dla prawego ogona |series|.
    
    Parameters
    ----------
    series : pd.Series
        Szereg zwrotów
    k_values : array-like
        Wartości k (liczba obserwacji w ogonie)
        
    Returns
    -------
    DataFrame z kolumnami: k, alpha, se
    """
    # Używamy wartości bezwzględnych (ogon prawostronny)
    x = np.sort(np.abs(series.dropna().values))[::-1]  # malejąco
    n = len(x)
    
    if k_values is None:
        # k od 10 do n//4
        k_values = range(10, n // 4, max(1, n // 200))
    
    results = []
    for k in k_values:
        if k >= n:
            break
        # Estymator Hilla
        log_ratios = np.log(x[:k] / x[k])
        alpha_inv = np.mean(log_ratios)  # 1/alpha
        alpha = 1.0 / alpha_inv if alpha_inv > 0 else np.nan
        # Błąd standardowy: se(alpha) ≈ alpha/sqrt(k)
        se = alpha / np.sqrt(k) if not np.isnan(alpha) else np.nan
        results.append({'k': k, 'alpha': alpha, 'se': se})
    
    return pd.DataFrame(results)

# Oblicz dla BTC i WIG20
btc_r_clean = df['btc_log_return'].dropna() * 100  # w procentach
wig_r_clean = df['wig_log_return'].dropna() * 100

print("Obliczanie estymatora Hilla...")
hill_btc = hill_estimator(btc_r_clean)
hill_wig = hill_estimator(wig_r_clean)

# Optymalny k (gdzie estymator się stabilizuje)
# Metoda: minimalizacja MSE lub median z zakresu k∈[50, n//8]
btc_n = len(btc_r_clean)
wig_n = len(wig_r_clean)
stable_k_range_btc = hill_btc[(hill_btc['k'] >= 50) & (hill_btc['k'] <= btc_n // 8)]
stable_k_range_wig = hill_wig[(hill_wig['k'] >= 50) & (hill_wig['k'] <= wig_n // 8)]

alpha_btc_opt = stable_k_range_btc['alpha'].median() if len(stable_k_range_btc) > 0 else np.nan
alpha_wig_opt = stable_k_range_wig['alpha'].median() if len(stable_k_range_wig) > 0 else np.nan

print(f"\nEstymat indeksu ogona alfa (Hill, mediana stabilnego zakresu):")
print(f"  BTC:   α = {alpha_btc_opt:.4f}")
print(f"  WIG20: α = {alpha_wig_opt:.4f}")

if not np.isnan(alpha_btc_opt):
    if alpha_btc_opt < 2:
        print(f"  BTC: BARDZO grube ogony (α < 2, nieskończona wariancja!)")
    elif alpha_btc_opt < 4:
        print(f"  BTC: Grube ogony (α ∈ (2,4), skończona wariancja, nieskończona kurtoza)")
    else:
        print(f"  BTC: Umiarkowane ogony (α > 4)")

# Wykres estymatora Hilla
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Estymator Hilla — Indeks ogona Pareto (α)', fontsize=13)

for ax, hill_df, name, color, alpha_opt in [
    (axes[0], hill_btc, 'BTC/USD', COLOR_BTC, alpha_btc_opt),
    (axes[1], hill_wig, 'WIG20', COLOR_WIG, alpha_wig_opt)
]:
    valid = hill_df.dropna()
    ax.plot(valid['k'], valid['alpha'], color=color, linewidth=1.2, label='α Hill')
    
    # Przedziały ufności (±1 SE)
    ax.fill_between(valid['k'],
                    valid['alpha'] - valid['se'],
                    valid['alpha'] + valid['se'],
                    alpha=0.2, color=color, label='±1 SE')
    
    # Linie referencyjne
    ax.axhline(2, color='red', linestyle='--', linewidth=0.8, label='α=2 (∞ wariancja)')
    ax.axhline(4, color='orange', linestyle='--', linewidth=0.8, label='α=4 (∞ kurtoza)')
    
    # Optymalny estymator
    if not np.isnan(alpha_opt):
        ax.axhline(alpha_opt, color='black', linestyle='-', linewidth=1.2,
                   label=f'α_opt = {alpha_opt:.3f}')
    
    ax.set_xlabel('k (liczba obserwacji ogona)')
    ax.set_ylabel('α (indeks ogona)')
    ax.set_title(f'{name} — Hill Estimator')
    ax.legend(fontsize=8)
    ax.set_ylim(0, 10)

plt.tight_layout()
for ext in ['png', 'pdf']:
    plt.savefig(RESULTS_FIGS / f'fig_09_hill_estimator.{ext}',
                dpi=300 if ext=='png' else 150, bbox_inches='tight')
plt.show()
print("Zapisano: fig_09_hill_estimator.png + .pdf")

# Zapis tabeli indeksów ogona
tail_df = pd.DataFrame([
    {'Szereg': 'BTC/USD Log-Return', 'Alpha Hill (mediana)': alpha_btc_opt,
     'Interpretacja': 'Grube ogony (α < 4)' if alpha_btc_opt < 4 else 'Umiarkowane ogony'},
    {'Szereg': 'WIG20 Log-Return', 'Alpha Hill (mediana)': alpha_wig_opt,
     'Interpretacja': 'Grube ogony (α < 4)' if alpha_wig_opt < 4 else 'Umiarkowane ogony'},
]).set_index('Szereg')

tail_df.to_csv(RESULTS_TABLES / '03_tail_index.csv', float_format='%.4f')
print("Zapisano: 03_tail_index.csv")

Obliczanie estymatora Hilla...

Estymat indeksu ogona alfa (Hill, mediana stabilnego zakresu):
  BTC:   α = 2.6995
  WIG20: α = 3.3617
  BTC: Grube ogony (α ∈ (2,4), skończona wariancja, nieskończona kurtoza)


Zapisano: fig_09_hill_estimator.png + .pdf
Zapisano: 03_tail_index.csv


## 5. Efekty ARCH i Model GARCH(1,1)

In [7]:
# ============================================================
# KROK 5: Test LM Engla (ARCH) + Model GARCH(1,1)
# ============================================================

print("=" * 70)
print("TEST ARCH (LM Engla)")
print("=" * 70)

arch_results = []
for series, name in [
    (df['btc_log_return'].dropna() * 100, 'BTC Log-Return (%)'),
    (df['wig_log_return'].dropna() * 100, 'WIG20 Log-Return (%)'),
]:
    # Test LM Engla na efekty ARCH
    lm_stat, lm_pval, f_stat, f_pval = het_arch(series, nlags=10)
    
    arch_results.append({
        'Szereg': name,
        'LM stat': round(lm_stat, 4),
        'LM p-value': round(lm_pval, 6),
        'F stat': round(f_stat, 4),
        'F p-value': round(f_pval, 6),
        'Efekty ARCH': 'TAK' if lm_pval < 0.05 else 'NIE'
    })
    print(f"\n{name}:")
    print(f"  LM stat = {lm_stat:.4f}, p-value = {lm_pval:.6f}")
    print(f"  Wniosek: Efekty ARCH {'TAK (GARCH uzasadniony)' if lm_pval < 0.05 else 'NIE'}")

arch_df = pd.DataFrame(arch_results).set_index('Szereg')
arch_df.to_csv(RESULTS_TABLES / '03_arch_test.csv', float_format='%.6f')

# ─── GARCH(1,1) ──────────────────────────────────────────────
print("\n" + "=" * 70)
print("MODEL GARCH(1,1)")
print("=" * 70)

garch_params_all = []
garch_models = {}

for series, name, color in [
    (df['btc_log_return'].dropna() * 100, 'BTC', COLOR_BTC),
    (df['wig_log_return'].dropna() * 100, 'WIG20', COLOR_WIG),
]:
    print(f"\nEstymacja GARCH(1,1) dla {name}...")
    
    # Usuń inf i nan
    series_clean = series.replace([np.inf, -np.inf], np.nan).dropna()
    
    try:
        model = arch_model(series_clean, vol='Garch', p=1, q=1, 
                           dist='t',  # rozkład t-Studenta (grube ogony)
                           mean='AR', lags=1)  # AR(1) dla średniej
        result = model.fit(disp='off', show_warning=False)
        
        print(result.summary())
        
        # Wyciągnij parametry
        params = result.params
        pvalues = result.pvalues
        
        omega = params.get('omega', np.nan)
        alpha1 = params.get('alpha[1]', np.nan)
        beta1 = params.get('beta[1]', np.nan)
        
        persistence = alpha1 + beta1 if not (np.isnan(alpha1) or np.isnan(beta1)) else np.nan
        
        garch_params_all.append({
            'Aktywo': name,
            'omega': round(omega, 6),
            'alpha[1]': round(alpha1, 6),
            'beta[1]': round(beta1, 6),
            'Persistence (a+b)': round(persistence, 6),
            'Log-likelihood': round(result.loglikelihood, 2),
            'AIC': round(result.aic, 2),
            'BIC': round(result.bic, 2)
        })
        
        garch_models[name] = result
        print(f"  alpha + beta = {persistence:.4f} ({'wysoka persistencja zmienności' if persistence > 0.95 else 'umiarkowana persistencja'})")
        
    except Exception as e:
        print(f"  Błąd estymacji GARCH dla {name}: {e}")

# Zapis tabeli parametrów GARCH
if garch_params_all:
    garch_df = pd.DataFrame(garch_params_all).set_index('Aktywo')
    garch_df.to_csv(RESULTS_TABLES / '03_garch_parameters.csv', float_format='%.6f')
    
    tex_path = RESULTS_TABLES / '03_garch_parameters.tex'
    garch_df.to_latex(
        str(tex_path), float_format='%.4f', escape=False,
        caption='Parametry modelu GARCH(1,1) z rozkładem t-Studenta',
        label='tab:garch_parameters'
    )
    print(f"\nZapisano parametry GARCH: {RESULTS_TABLES / '03_garch_parameters.csv'}")

TEST ARCH (LM Engla)

BTC Log-Return (%):
  LM stat = 33.0132, p-value = 0.000271
  Wniosek: Efekty ARCH TAK (GARCH uzasadniony)

WIG20 Log-Return (%):
  LM stat = 285.0123, p-value = 0.000000
  Wniosek: Efekty ARCH TAK (GARCH uzasadniony)

MODEL GARCH(1,1)

Estymacja GARCH(1,1) dla BTC...
                              AR - GARCH Model Results                              
Dep. Variable:               btc_log_return   R-squared:                       0.002
Mean Model:                              AR   Adj. R-squared:                  0.002
Vol Model:                            GARCH   Log-Likelihood:               -4991.03
Distribution:      Standardized Student's t   AIC:                           9994.05
Method:                  Maximum Likelihood   BIC:                           10027.4
                                              No. Observations:                 1904
Date:                      Tue, Aug 18 2026   Df Residuals:                     1902
Time:                        

In [8]:
# ============================================================
# Wykres: warunkowa zmienność GARCH
# ============================================================

if len(garch_models) > 0:
    fig, axes = plt.subplots(2, 1, figsize=(14, 10), sharex=False)
    fig.suptitle('Warunkowa zmienność GARCH(1,1) — BTC i WIG20', fontsize=13)
    
    for i, (name, result) in enumerate(garch_models.items()):
        ax = axes[i]
        color = COLOR_BTC if name == 'BTC' else COLOR_WIG
        
        # Warunkowe odchylenie standardowe
        cond_vol = result.conditional_volatility
        
        ax.plot(cond_vol.index, cond_vol, color=color, linewidth=0.8, alpha=0.9,
                label=f'{name} warunkowa zmienność')
        ax.fill_between(cond_vol.index, cond_vol, alpha=0.15, color=color)
        
        # Zaznacz zdarzenia kryzysowe
        crisis_events = {
            '2020-03-12': ('COVID', '#E74C3C'),
            '2022-05-09': ('Terra', '#E67E22'),
            '2022-11-08': ('FTX', '#8E44AD'),
            '2023-03-10': ('SVB', '#2980B9'),
        }
        for date_str, (label, ev_color) in crisis_events.items():
            event_date = pd.Timestamp(date_str)
            if cond_vol.index.min() <= event_date <= cond_vol.index.max():
                ax.axvline(event_date, color=ev_color, alpha=0.7, linewidth=1.2, linestyle='--')
                ax.text(event_date + pd.Timedelta(days=3),
                        cond_vol.max() * 0.92, label,
                        rotation=90, fontsize=8, color=ev_color, va='top')
        
        # Statystyki
        ax.text(0.98, 0.97,
                f'α+β = {garch_params_all[i]["Persistence (a+b)"]:.4f}',
                transform=ax.transAxes, fontsize=9, ha='right', va='top',
                bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
        
        ax.set_ylabel(f'{name} — cond. vol (%)')
        ax.set_title(f'{name} — GARCH(1,1) warunkowa zmienność')
        ax.legend(fontsize=9)
        ax.xaxis.set_major_formatter(mpl.dates.DateFormatter('%Y-%m'))
        ax.xaxis.set_major_locator(mpl.dates.MonthLocator(interval=6))
        plt.setp(ax.xaxis.get_majorticklabels(), rotation=45, ha='right')
    
    plt.tight_layout()
    for ext in ['png', 'pdf']:
        plt.savefig(RESULTS_FIGS / f'fig_10_garch_volatility.{ext}',
                    dpi=300 if ext=='png' else 150, bbox_inches='tight')
    plt.show()
    print("Zapisano: fig_10_garch_volatility.png + .pdf")

Zapisano: fig_10_garch_volatility.png + .pdf


## 6. Kointegracja Johansena i Przyczynowość Grangera

In [9]:
# ============================================================
# KROK 6: Kointegracja Johansena
# ============================================================
# Testuje czy BTC i WIG20 mają długookresowy związek kointegrujący.
# H0: rank kointegracji = 0 (brak kointegracji)

print("=" * 70)
print("TEST KOINTEGRACJI JOHANSENA — BTC vs WIG20")
print("=" * 70)

# Przygotowanie danych (log-ceny, wspólny indeks)
log_btc = np.log(df['btc_close'])
log_wig = np.log(df['wig_close'])

# Wyrównaj indeksy
common_idx = log_btc.dropna().index.intersection(log_wig.dropna().index)
johansen_data = pd.DataFrame({
    'log_btc': log_btc.loc[common_idx],
    'log_wig': log_wig.loc[common_idx]
}).dropna()

print(f"Dane do testu Johansena: {johansen_data.shape}")

try:
    # det_order: 0 = bez stałej, 1 = stała, -1 = bez deterministycznych
    johanssen_result = coint_johansen(johansen_data, det_order=0, k_ar_diff=2)
    
    # Trace statistics
    print("\nTrace Statistics (H0: rank ≤ r):")
    trace_stats = johanssen_result.lr1
    trace_cv = johanssen_result.cvt  # Critical values [90%, 95%, 99%]
    
    johansen_rows = []
    for i, (ts, cv) in enumerate(zip(trace_stats, trace_cv)):
        rejected_5pct = ts > cv[1]  # 95% level
        johansen_rows.append({
            'H0: rank ≤': i,
            'Trace stat': round(ts, 4),
            'CV 95%': round(cv[1], 4),
            'Odrzucenie H0 (5%)': 'TAK' if rejected_5pct else 'NIE'
        })
        print(f"  r ≤ {i}: Trace = {ts:.4f}, CV 95% = {cv[1]:.4f} → {'Odrzuć H0' if rejected_5pct else 'Nie odrzucaj H0'}")
    
    johansen_df = pd.DataFrame(johansen_rows).set_index('H0: rank ≤')
    johansen_df.to_csv(RESULTS_TABLES / '03_johansen_cointegration.csv', float_format='%.4f')
    
    # Max eigenvalue statistics
    print("\nMax Eigenvalue Statistics:")
    max_stats = johanssen_result.lr2
    max_cv = johanssen_result.cvm
    for i, (ms, cv) in enumerate(zip(max_stats, max_cv)):
        rejected_5pct = ms > cv[1]
        print(f"  r = {i}: Max Eig = {ms:.4f}, CV 95% = {cv[1]:.4f} → {'Odrzuć H0' if rejected_5pct else 'Nie odrzucaj H0'}")
    
    n_cointegrating = sum(1 for ts, cv in zip(trace_stats, trace_cv) if ts > cv[1])
    print(f"\nWniosek: Liczba wektorów kointegrujących = {n_cointegrating}")
    if n_cointegrating == 0:
        print("  Brak kointegracji długookresowej między BTC i WIG20.")
        print("  Transmisja ryzyka ma charakter krótkookresowy (np. przez kanał płynności).")
    else:
        print(f"  Istnieje {n_cointegrating} wektor(y) kointegrujący — długookresowe powiązanie.")
    
except Exception as e:
    print(f"Błąd testu Johansena: {e}")
    print("Próba testu Engle-Granger (alternatywny):")
    try:
        score, pvalue, _ = coint(log_btc.dropna(), log_wig.dropna())
        print(f"  Engle-Granger stat = {score:.4f}, p-value = {pvalue:.4f}")
        print(f"  Wniosek: {'Kointegracja' if pvalue < 0.05 else 'Brak kointegracji'}")
    except Exception as e2:
        print(f"  Błąd: {e2}")

print("\nZapisano: 03_johansen_cointegration.csv")

TEST KOINTEGRACJI JOHANSENA — BTC vs WIG20
Dane do testu Johansena: (1906, 2)

Trace Statistics (H0: rank ≤ r):
  r ≤ 0: Trace = 9.0136, CV 95% = 15.4943 → Nie odrzucaj H0
  r ≤ 1: Trace = 0.5274, CV 95% = 3.8415 → Nie odrzucaj H0

Max Eigenvalue Statistics:
  r = 0: Max Eig = 8.4862, CV 95% = 14.2639 → Nie odrzucaj H0
  r = 1: Max Eig = 0.5274, CV 95% = 3.8415 → Nie odrzucaj H0

Wniosek: Liczba wektorów kointegrujących = 0
  Brak kointegracji długookresowej między BTC i WIG20.
  Transmisja ryzyka ma charakter krótkookresowy (np. przez kanał płynności).

Zapisano: 03_johansen_cointegration.csv


In [10]:
# ============================================================
# KROK 7: Test przyczynowości Grangera
# ============================================================
# Testuje czy przeszłe wartości BTC pomagają prognozować WIG20
# (i odwrotnie). Test wymaga stacjonarności — używamy log-returns.

print("=" * 70)
print("TEST PRZYCZYNOWOŚCI GRANGERA")
print("=" * 70)

# Przygotowanie: wyrównaj indeksy
granger_df = pd.DataFrame({
    'btc': df['btc_log_return'],
    'wig': df['wig_log_return']
}).dropna()

print(f"Dane Grangera: {granger_df.shape}")

max_lag = 10
granger_results = []

# Test: BTC → WIG20 (czy BTC Granger-przyczynuje WIG20?)
print("\n[BTC → WIG20] H0: BTC NIE Granger-przyczynuje WIG20")
print("-" * 50)

try:
    gc_btc_wig = grangercausalitytests(granger_df[['wig', 'btc']], maxlag=max_lag,
                                        verbose=False)
    for lag in range(1, max_lag + 1):
        f_stat = gc_btc_wig[lag][0]['ssr_ftest'][0]
        p_val = gc_btc_wig[lag][0]['ssr_ftest'][1]
        granger_results.append({
            'Kierunek': 'BTC → WIG20',
            'Lag': lag,
            'F stat': round(f_stat, 4),
            'p-value': round(p_val, 6),
            'Odrzucenie H0 (5%)': 'TAK' if p_val < 0.05 else 'NIE'
        })
        print(f"  Lag {lag:2d}: F = {f_stat:.4f}, p = {p_val:.4f} {'***' if p_val < 0.01 else '**' if p_val < 0.05 else '.'  }")
except Exception as e:
    print(f"Błąd testu Grangera BTC→WIG: {e}")

# Test: WIG20 → BTC (sprawdzenie odwrotnego kierunku)
print("\n[WIG20 → BTC] H0: WIG20 NIE Granger-przyczynuje BTC")
print("-" * 50)

try:
    gc_wig_btc = grangercausalitytests(granger_df[['btc', 'wig']], maxlag=max_lag,
                                        verbose=False)
    for lag in range(1, max_lag + 1):
        f_stat = gc_wig_btc[lag][0]['ssr_ftest'][0]
        p_val = gc_wig_btc[lag][0]['ssr_ftest'][1]
        granger_results.append({
            'Kierunek': 'WIG20 → BTC',
            'Lag': lag,
            'F stat': round(f_stat, 4),
            'p-value': round(p_val, 6),
            'Odrzucenie H0 (5%)': 'TAK' if p_val < 0.05 else 'NIE'
        })
        print(f"  Lag {lag:2d}: F = {f_stat:.4f}, p = {p_val:.4f} {'***' if p_val < 0.01 else '**' if p_val < 0.05 else '.'  }")
except Exception as e:
    print(f"Błąd testu Grangera WIG→BTC: {e}")

# Zapis wyników
granger_df_out = pd.DataFrame(granger_results)
if len(granger_df_out) > 0:
    granger_pivot = granger_df_out.pivot_table(values='p-value', index='Lag', columns='Kierunek')
    granger_pivot.to_csv(RESULTS_TABLES / '03_granger_causality.csv', float_format='%.6f')
    
    granger_pivot.to_latex(
        str(RESULTS_TABLES / '03_granger_causality.tex'),
        float_format='%.4f', escape=False,
        caption='p-wartości testu przyczynowości Grangera (lagi 1--10)',
        label='tab:granger_causality'
    )
    print(f"\nZapisano: {RESULTS_TABLES / '03_granger_causality.csv'}")
    
    # Wizualizacja p-values
    fig, ax = plt.subplots(figsize=(12, 5))
    
    for direction, color, marker in [
        ('BTC → WIG20', COLOR_BTC, 'o'),
        ('WIG20 → BTC', COLOR_WIG, 's')
    ]:
        subset = granger_df_out[granger_df_out['Kierunek'] == direction]
        if len(subset) > 0:
            ax.plot(subset['Lag'], subset['p-value'],
                    color=color, marker=marker, linewidth=1.5, markersize=7,
                    label=f'{direction}')
            # Oznacz istotne lagi
            sig = subset[subset['p-value'] < 0.05]
            ax.scatter(sig['Lag'], sig['p-value'], color=color, s=100, zorder=5,
                       edgecolors='black', linewidth=1)
    
    ax.axhline(0.05, color='red', linestyle='--', linewidth=1.2, label='α=0.05')
    ax.axhline(0.01, color='darkred', linestyle=':', linewidth=1.0, label='α=0.01')
    ax.set_xlabel('Lag')
    ax.set_ylabel('p-value')
    ax.set_title('Przyczynowość Grangera: BTC ↔ WIG20\n'
                 'p-wartości testu F (lagi 1-10)')
    ax.legend(fontsize=9)
    ax.set_xticks(range(1, max_lag + 1))
    ax.set_ylim(0, 1)
    
    plt.tight_layout()
    for ext in ['png', 'pdf']:
        plt.savefig(RESULTS_FIGS / f'fig_11_granger_pvalues.{ext}',
                    dpi=300 if ext=='png' else 150, bbox_inches='tight')
    plt.show()
    print("Zapisano: fig_11_granger_pvalues.png")

TEST PRZYCZYNOWOŚCI GRANGERA
Dane Grangera: (1905, 2)

[BTC → WIG20] H0: BTC NIE Granger-przyczynuje WIG20
--------------------------------------------------
  Lag  1: F = 2.4147, p = 0.1204 .
  Lag  2: F = 2.8836, p = 0.0562 .
  Lag  3: F = 2.4603, p = 0.0610 .
  Lag  4: F = 2.7349, p = 0.0275 **
  Lag  5: F = 2.4605, p = 0.0312 **
  Lag  6: F = 2.0473, p = 0.0565 .
  Lag  7: F = 2.0790, p = 0.0427 **
  Lag  8: F = 1.8317, p = 0.0670 .
  Lag  9: F = 1.6856, p = 0.0872 .
  Lag 10: F = 1.5844, p = 0.1052 .

[WIG20 → BTC] H0: WIG20 NIE Granger-przyczynuje BTC
--------------------------------------------------
  Lag  1: F = 0.1657, p = 0.6840 .
  Lag  2: F = 0.8856, p = 0.4126 .
  Lag  3: F = 0.5882, p = 0.6227 .
  Lag  4: F = 0.8614, p = 0.4864 .
  Lag  5: F = 0.7516, p = 0.5848 .
  Lag  6: F = 0.6371, p = 0.7007 .
  Lag  7: F = 0.6477, p = 0.7166 .
  Lag  8: F = 0.9137, p = 0.5039 .
  Lag  9: F = 0.8868, p = 0.5363 .
  Lag 10: F = 0.9579, p = 0.4786 .

Zapisano: results/tables/03_grange

Zapisano: fig_11_granger_pvalues.png


In [11]:
# ============================================================
# PODSUMOWANIE NOTEBOOKA 03
# ============================================================

print("=" * 60)
print(f"Notebook 03 zakończony: {pd.Timestamp.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("=" * 60)

saved_files = [
    RESULTS_TABLES / '03_stationarity_tests.csv',
    RESULTS_TABLES / '03_stationarity_tests.tex',
    RESULTS_FIGS / 'fig_08_acf_pacf.png',
    RESULTS_TABLES / '03_ljungbox_test.csv',
    RESULTS_TABLES / '03_normality_tests.csv',
    RESULTS_TABLES / '03_tail_index.csv',
    RESULTS_FIGS / 'fig_09_hill_estimator.png',
    RESULTS_TABLES / '03_arch_test.csv',
    RESULTS_TABLES / '03_garch_parameters.csv',
    RESULTS_FIGS / 'fig_10_garch_volatility.png',
    RESULTS_TABLES / '03_johansen_cointegration.csv',
    RESULTS_TABLES / '03_granger_causality.csv',
    RESULTS_FIGS / 'fig_11_granger_pvalues.png',
]

print("\nZapisane pliki:")
for f in saved_files:
    exists = f.exists()
    status = 'OK' if exists else 'BRAK'
    size = f"{f.stat().st_size / 1024:.1f} KB" if exists else '---'
    print(f"  [{status}] {f.name:<50} ({size})")

print("\n" + "=" * 60)
print("GOTOWE! Przejdź do Notebook 04 — Feature Engineering.")
print("=" * 60)

Notebook 03 zakończony: 2026-08-18 20:33:42

Zapisane pliki:
  [OK] 03_stationarity_tests.csv                          (0.8 KB)
  [OK] 03_stationarity_tests.tex                          (0.8 KB)
  [OK] fig_08_acf_pacf.png                                (441.0 KB)
  [OK] 03_ljungbox_test.csv                               (0.3 KB)
  [OK] 03_normality_tests.csv                             (0.4 KB)
  [OK] 03_tail_index.csv                                  (0.1 KB)
  [OK] fig_09_hill_estimator.png                          (238.4 KB)
  [OK] 03_arch_test.csv                                   (0.2 KB)
  [OK] 03_garch_parameters.csv                            (0.2 KB)
  [OK] fig_10_garch_volatility.png                        (659.2 KB)
  [OK] 03_johansen_cointegration.csv                      (0.1 KB)
  [OK] 03_granger_causality.csv                           (0.2 KB)
  [OK] fig_11_granger_pvalues.png                         (170.2 KB)

GOTOWE! Przejdź do Notebook 04 — Feature Engineering.
